In [1]:
pip install python-flint

Note: you may need to restart the kernel to use updated packages.


In [2]:
import flint


In [3]:
from flint import arb,ctx

In [24]:
print(f"python-flint version = {flint.__version__}")

python-flint version = 0.9.0


The next two examples are Arb failures, I think. I have reported them to https://github.com/flintlib/flint/issues/2964

In [26]:
acb(0.7523359028935359, - 0.658779697028808).hypgeom_2f1(acb(37), acb(12), acb(48))

[+/- 3.35e+4] + [+/- 3.35e+4]j

In [28]:
acb(0.7298184519936511 , 0.6836410074955939). hypgeom_2f1(acb(32), acb(14), acb(48))

[+/- 4.11e+21] + [+/- 4.11e+21]j

In [27]:
acb(0.7523359028935359, - 0.658779697028808).hypgeom_2f1(acb(2), acb(12), acb(48))

[1.3256142 +/- 4.51e-8] + [-0.5862414 +/- 4.62e-8]j

In [5]:
from math import ceil, log10
from flint import acb, ctx


def bits_to_digits(bits, guard=5):
    """
    Convert binary precision to a reasonable Maxima fpprec.
    """
    return ceil(bits * log10(2)) + guard


def clean_flint(s):
    """
    Remove Flint interval decorations.
    """
    s = s.split("+/-")[0].strip()
    s = s.lstrip("[").rstrip("]")
    return s


def maxima_bigfloat_string(s):
    """
    Convert scientific notation to Maxima bigfloat notation.
    """

    s = clean_flint(s)

    if "e" in s:
        mant, exp = s.split("e")
        return f"{mant}b{exp}"

    if "E" in s:
        mant, exp = s.split("E")
        return f"{mant}b{exp}"

    if "b" not in s:
        s += "b0"

    return s


def maximaify(z, use_bigfloats=False, digits=50):
    """
    Convert a Flint acb value to Maxima syntax.
    """

    re = clean_flint(z.real.str(digits))
    im = clean_flint(z.imag.str(digits))

    if use_bigfloats:
        re = maxima_bigfloat_string(re)
        im = maxima_bigfloat_string(im)

    if im in ("0", "0.0", "0b0"):
        return re

    if im.startswith("-"):
        return f"({re} - {im[1:]}*%i)"

    return f"({re} + {im}*%i)"


def maxima_arg(x, use_bigfloats=False):
    """
    Convert a Python value into Maxima syntax.
    """

    if isinstance(x, complex):

        re = str(x.real)
        im = str(abs(x.imag))

        if use_bigfloats:
            re = maxima_bigfloat_string(re)
            im = maxima_bigfloat_string(im)

        sign = "+" if x.imag >= 0 else "-"

        return f"({re} {sign} {im}*%i)"

    s = str(x)

    if use_bigfloats:
        return maxima_bigfloat_string(s)

    return s


def print_test(a, b, c, x, bits=53):
    """
    Generate a Maxima test of the form

        boa(...);
        true$

    bits=53  -> ordinary float computation

    bits!=53 -> bigfloat computation using fpprec
    """

    work_digits = bits_to_digits(max(200, bits + 50))
    ctx.dps = work_digits

    value = acb(x).hypgeom_2f1(acb(a), acb(b), acb(c))

    if bits == 53:

        aa = maxima_arg(a)
        bb = maxima_arg(b)
        cc = maxima_arg(c)
        xx = maxima_arg(x)

        expected = maximaify(
            value,
            use_bigfloats=False,
            digits=work_digits
        )

        print(
            f"boa(gauss_hg({aa}, {bb}, {cc}, {xx}), "
            f"{expected}, {bits});"
        )

    else:

        aa = maxima_arg(a, True)
        bb = maxima_arg(b, True)
        cc = maxima_arg(c, True)
        xx = maxima_arg(x, True)

        expected = maximaify(
            value,
            use_bigfloats=True,
            digits=work_digits
        )

        fpprec = bits_to_digits(bits)

        print(
            f"boa(block([fpprec:{fpprec}], "
            f"gauss_hg({aa}, {bb}, {cc}, {xx})), "
            f"{expected}, {bits});"
        )

    print("true$")
    print()

In [6]:
print_test(
    2.1,
    4.1,
    8.7,
    -10.2, 53)

boa(gauss_hg(2.1, 4.1, 8.7, -10.2), 0.03463804379393735258560929627194193646430881415960, 53);
true$



In [7]:
print_test(2,40,8000, -10.2)

boa(gauss_hg(2, 40, 8000, -10.2), 0.90546286684862023890027539394601730178129031634102734757029418, 53);
true$



In [8]:
print_test(2, 40, 8, 0.8)

boa(gauss_hg(2, 40, 8, 0.8), 4547450558531340952.753504114510885301739899569525054889377387875, 53);
true$



In [9]:
print_test(2,4,8, -5.2)

boa(gauss_hg(2, 4, 8, -5.2), 0.095078687314954104589466234089029908979887325818865299648298355, 53);
true$



In [10]:
print_test(5, 2, 400, 2+1.1j)

boa(gauss_hg(5, 2, 400, (2.0 + 1.1*%i)), (nan + nan*%i), 53);
true$



In [11]:
acb(0,1.1).hypgeom_2f1(acb(5), acb(2), acb(400))

nan + nanj

In [12]:
print_test(5,2,400, 1.1j)

boa(gauss_hg(5, 2, 400, (0.0 + 1.1*%i)), (nan + nan*%i), 53);
true$



In [13]:
import random

In [15]:
def generate_tests_on_minus_one_one(n=50, seed=12345):
    random.seed(seed)

    for _ in range(n):
        print_test(
            random.randint(1, 50),
            random.randint(1, 50),
            random.randint(1, 50),
            random.uniform(-0.99, 0.99)
        )

In [19]:
import random
import math

def generate_tests_in_unit_circle(n=50, seed=12345):
    random.seed(seed)

    for _ in range(n):
        theta = random.uniform(0.0, 2.0 * math.pi)
        r = math.sqrt(random.random())  # uniform in unit disk
        x = r * complex(math.cos(theta), math.sin(theta))

        print_test(
            random.randint(1, 50),
            random.randint(1, 50),
            random.randint(1, 50),
            x
        )
       

In [22]:
import random
import math

def generate_tests_on_unit_circle(n=50, seed=12345):
    random.seed(seed)

    for _ in range(n):
        theta = random.uniform(0.0, 2.0 * math.pi)
        x = complex(math.cos(theta), math.sin(theta))

        print_test(
            random.randint(1, 50),
            random.randint(1, 50),
            random.randint(1, 50),
            x
        )

In [29]:
import random
import math

def generate_tests_annulus(radius1, radius2, n=50, seed=12345):
    if radius1 <= 0:
        raise ValueError("radius1 must be positive")
    if radius1 >= radius2:
        raise ValueError("require radius1 < radius2")

    random.seed(seed)

    for _ in range(n):
        theta = random.uniform(0.0, 2.0 * math.pi)

        # Uniform with respect to area in the annulus
        u = random.random()
        r = math.sqrt(radius1**2 + u * (radius2**2 - radius1**2))

        x = r * complex(math.cos(theta), math.sin(theta))

        print_test(
            random.randint(1, 50),
            random.randint(1, 50),
            random.randint(1, 50),
            x
        )

In [30]:
generate_tests_annulus(1,25, 100)

boa(gauss_hg(20, 24, 13, (-2.3467638614934114 + 1.3558248853001593*%i)), (-2.46178682693999660559932494489073606153608096047446360446e-17 - 1.69246141323471089348487260809125033580174875752677253071e-17*%i), 53);
true$

boa(gauss_hg(24, 8, 28, (-2.096852920098903 + 16.395418142112995*%i)), (-3.417055953985683869741325583671277074264198222783643120323422673e-11 - 7.325341729266017605219383028028549745081129701072067736477569800e-10*%i), 53);
true$

boa(gauss_hg(40, 36, 12, (-1.379159310599687 + 19.760564561594673*%i)), (3.0043405360575050369988018971274308746284974370488e-56 + 4.070811156101838595442202192193310979163818452929e-56*%i), 53);
true$

boa(gauss_hg(6, 34, 48, (-14.986497310362848 + 19.345214505624007*%i)), (1.6404566865392966199490306939978925754299874323891390445710313330e-8 - 3.187388531539893057022535984417124560808712112941806266585186243e-8*%i), 53);
true$

boa(gauss_hg(10, 14, 47, (-15.121567111190087 + 9.31632319117567*%i)), (-7.951522530691490428669937334321343667517